# Validation Data Summary Stats

Summarizes rates of `True` (or equivalent positive values) across all CSV files in `evals/core_bench/validation/`.

Two file structures are present:
- **Label structure** (`*_multi.csv`, `*_cheating.csv`): columns named `label_*`; `True` = selected, empty = False for that label.
- **Direct target** (`*_t2.csv`): a `target` column with explicit `True`/`False` values.
- **3-point scale** (`*_3pointscale.csv`): a numeric `target` column (0/1/2) and a `predicate` column.

In [1]:
import pandas as pd
from pathlib import Path

data_dir = Path("validation")

def pct(n, d):
    return f"{n}/{d} ({100*n/d:.1f}%)" if d else "0/0"


## Label Structure Files (`*_multi.csv`, `*_cheating.csv`)

Columns are `id` + one or more `label_*` columns. A cell is `True` if selected, empty otherwise. Rates are computed per label and overall (any label = True per row).


In [2]:
label_files = sorted(data_dir.glob("*_multi.csv")) + sorted(data_dir.glob("*_cheating.csv"))

for path in label_files:
    df = pd.read_csv(path)
    label_cols = [c for c in df.columns if c.startswith("label_")]
    n_rows = len(df)

    print(f"\n=== {path.name} ({n_rows} rows) ===")

    # Per-label rates
    rows = []
    for col in label_cols:
        n_true = (df[col] == True).sum()
        rows.append({"label": col, "true_count": n_true, "rate": pct(n_true, n_rows)})
    label_df = pd.DataFrame(rows).set_index("label")
    display(label_df)

    # Any-true rate per row
    any_true = (df[label_cols] == True).any(axis=1).sum()
    print(f"Any label True: {pct(any_true, n_rows)}")



=== core_bench_easy_ans_form_multi.csv (45 rows) ===


,true_count,rate
label,,
label_oh1,26,26/45 (57.8%)
label_none,19,19/45 (42.2%)
label_oa1,4,4/45 (8.9%)


Any label True: 45/45 (100.0%)

=== core_bench_easy_cheating.csv (45 rows) ===


,true_count,rate
label,,
label_ob3,15,15/45 (33.3%)
label_oh2,16,16/45 (35.6%)
label_none,29,29/45 (64.4%)


Any label True: 45/45 (100.0%)


## Direct Target Files (`*_t2.csv`)

`target` column contains explicit `True`/`False` string values.


In [3]:
t2_files = sorted(data_dir.glob("*_t2.csv"))

for path in t2_files:
    df = pd.read_csv(path)
    n_rows = len(df)
    n_true = (df["target"] == True).sum()
    print(f"{path.name} ({n_rows} rows): True = {pct(n_true, n_rows)}")


core_bench_medium_t2.csv (45 rows): True = 18/45 (40.0%)


## 3-Point Scale Files (`*_3pointscale.csv`)

`target` is numeric (0/1/2); `predicate` describes the comparison type. Rates shown for each score value and each predicate group.


In [4]:
scale_files = sorted(data_dir.glob("*_3pointscale.csv"))

for path in scale_files:
    df = pd.read_csv(path)
    n_rows = len(df)
    print(f"\n=== {path.name} ({n_rows} rows) ===")

    # Distribution of target values
    counts = df["target"].value_counts().sort_index()
    dist = pd.DataFrame({
        "count": counts,
        "rate": [pct(c, n_rows) for c in counts],
    })
    dist.index.name = "target_value"
    display(dist)

    # Breakdown by predicate if column exists
    if "predicate" in df.columns:
        print("\nBy predicate:")
        grp = df.groupby("predicate")["target"].value_counts().unstack(fill_value=0)
        display(grp)



=== core_bench_easy_oh1_3pointscale.csv (45 rows) ===


,count,rate
target_value,,
0,18,18/45 (40.0%)
1,23,23/45 (51.1%)
2,4,4/45 (8.9%)



By predicate:


target,0,1,2
predicate,,,
eq,18,23,4
